In [ ]:
# ==============================================================
# 02 – Baseline Models (Multi-Country + Cost-Sensitive)
# Title: Multi-Agent DRL + CNN Alternative Data for Credit Decisioning
# Supports: RQ1 (baseline comparison) and RQ4 (business metrics)
# ==============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
%matplotlib inline

# --------------------------------------------------------------
# Paths
# --------------------------------------------------------------
ROOT = Path(".")
DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

print("Baseline Models Notebook – Multi-Country Version")

# --------------------------------------------------------------
# 1. Load Data (prefer global multi-country, fallback to German)
# --------------------------------------------------------------
global_path = DATA_SYNTHETIC / "global_credit_from_german.csv"
german_path = DATA_PROCESSED / "german_credit_processed.csv"

if global_path.exists():
    df = pd.read_csv(global_path)
    print(f"Loaded multi-country dataset: {df.shape}")
else:
    df = pd.read_csv(german_path)
    print(f"Loaded German dataset: {df.shape}")

print(f"Default rate: {df['default'].mean():.2%}")
print(f"Thin-file rate: {df['thin_file'].mean():.2%}")

# --------------------------------------------------------------
# 2. Feature Preparation
# --------------------------------------------------------------
# Select usable features
num_cols = ['duration', 'credit_amount', 'installment_rate', 'residence_since',
            'age', 'existing_credits', 'num_dependents']

# Some columns may have different names after multi-country generation
if 'income_proxy' in df.columns:
    num_cols.append('income_proxy')

cat_cols = ['checking_status', 'credit_history', 'purpose', 'savings',
            'employment', 'personal_status_sex', 'property', 'housing', 'job']

# Keep only columns that actually exist
num_cols = [c for c in num_cols if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]

# Simple encoding for categorical
df_encoded = df[num_cols + ['thin_file', 'default']].copy()

for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col].astype(str))

# Add country if available
if 'country' in df.columns:
    df_encoded = pd.concat([df_encoded, pd.get_dummies(df['country'], prefix='country')], axis=1)

feature_cols = [c for c in df_encoded.columns if c != 'default']
X = df_encoded[feature_cols].values.astype(np.float32)
y = df_encoded['default'].values.astype(np.int64)
thin = df_encoded['thin_file'].values.astype(np.int64)

print(f"Feature matrix: {X.shape}")

# --------------------------------------------------------------
# 3. Train / Test Split
# --------------------------------------------------------------
X_train, X_test, y_train, y_test, thin_train, thin_test = train_test_split(
    X, y, thin, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}")

# --------------------------------------------------------------
# 4. Cost-Sensitive Setup (important for credit)
# --------------------------------------------------------------
# Higher penalty for False Negatives (approving a bad customer)
# cost_fn = 5, cost_fp = 1  →  class_weight roughly {0:1, 1:5}
class_weight = {0: 1.0, 1: 5.0}

# --------------------------------------------------------------
# 5. Define Baseline Models
# --------------------------------------------------------------
baselines = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight=class_weight, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight=class_weight,
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=4, random_state=42
    ),
    "XGBoost (Cost-Sensitive)": xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        scale_pos_weight=5.0,           # cost-sensitive
        random_state=42, eval_metric='logloss', n_jobs=-1
    )
}

# --------------------------------------------------------------
# 6. Train & Evaluate
# --------------------------------------------------------------
results = []

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, thin_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else y_pred

    # Overall metrics
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred, zero_division=0)
    f1 = f1_score(y_te, y_pred, zero_division=0)
    auc = roc_auc_score(y_te, y_prob)

    # Thin-file specific performance (RQ4)
    thin_mask = thin_te == 1
    if thin_mask.sum() > 0:
        thin_rec = recall_score(y_te[thin_mask], y_pred[thin_mask], zero_division=0)
        thin_acc = accuracy_score(y_te[thin_mask], y_pred[thin_mask])
    else:
        thin_rec, thin_acc = np.nan, np.nan

    return {
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC-AUC": auc,
        "Thin-file Recall": thin_rec,
        "Thin-file Accuracy": thin_acc
    }

print("\nTraining baselines...")
for name, model in baselines.items():
    res = evaluate_model(name, model, X_train_scaled, X_test_scaled, y_train, y_test, thin_test)
    results.append(res)
    print(f"✓ {name:25} | AUC: {res['ROC-AUC']:.4f} | Thin-file Recall: {res['Thin-file Recall']:.4f}")

results_df = pd.DataFrame(results)
print("\n=== Baseline Results ===")
display(results_df.round(4))

# Save results
results_df.to_csv(RESULTS / "baseline_results.csv", index=False)
joblib.dump(scaler, RESULTS / "baseline_scaler.joblib")

# Save best model (highest AUC)
best_idx = results_df["ROC-AUC"].idxmax()
best_name = results_df.loc[best_idx, "Model"]
best_model = baselines[best_name]
joblib.dump(best_model, RESULTS / "best_baseline_model.joblib")
print(f"\nBest baseline: {best_name} (AUC = {results_df.loc[best_idx, 'ROC-AUC']:.4f})")

# --------------------------------------------------------------
# 7. Visualization
# --------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC comparison
sns.barplot(data=results_df, x="ROC-AUC", y="Model", ax=axes[0], palette="viridis")
axes[0].set_title("ROC-AUC Comparison (Higher is Better)")
axes[0].set_xlim(0.5, 1.0)

# Thin-file Recall
sns.barplot(data=results_df, x="Thin-file Recall", y="Model", ax=axes[1], palette="rocket")
axes[1].set_title("Thin-file Recall (Important for Inclusion – RQ4)")
axes[1].set_xlim(0, 1.0)

plt.tight_layout()
plt.savefig(RESULTS / "baseline_comparison.png", dpi=140, bbox_inches="tight")
plt.show()

print("\n✅ Baseline Models completed.")
print("Results saved to results/baseline_results.csv")
print("Next → 03_CNN_Encoder.ipynb (for RQ2)")